# exp_015g Smoke Submit

Minimal Kaggle code-competition smoke test.

Goals:
- verify the notebook can see competition data
- verify the run mode does not timeout before a trivial `submission.csv`
- separate environment issues from model issues before retrying heavier submit paths


In [ ]:
from __future__ import annotations

import json
import os
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd

_WALL_START = time.time()
INPUT_ROOT = Path("/kaggle/input")
WORKDIR = Path("/kaggle/working")
COMPETITION_HINT = "birdclef-2026"

print("cwd:", Path.cwd())
print("python:", platform.python_version())
print("platform:", platform.platform())
print("KAGGLE_KERNEL_RUN_TYPE:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print("KAGGLE_URL_BASE:", os.environ.get("KAGGLE_URL_BASE"))
print("torch CUDA visible env:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("input root exists:", INPUT_ROOT.exists())
print("working dir:", WORKDIR)


In [ ]:
def resolve_competition_dir(hint: str | None = None) -> Path:
    candidates = []
    for path in INPUT_ROOT.rglob("sample_submission.csv"):
        if hint and hint.lower() not in str(path).lower():
            continue
        candidates.append(path.parent)
    if not candidates:
        raise FileNotFoundError("Could not find sample_submission.csv under /kaggle/input")
    candidates = sorted(candidates, key=lambda p: (hint is None or hint.lower() not in str(p).lower(), len(str(p))))
    return candidates[0]

COMPETITION_DIR = resolve_competition_dir(COMPETITION_HINT)
SAMPLE_SUB_PATH = COMPETITION_DIR / "sample_submission.csv"
if not SAMPLE_SUB_PATH.exists():
    raise FileNotFoundError(f"Missing sample_submission.csv at {SAMPLE_SUB_PATH}")

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
print("Competition dir:", COMPETITION_DIR)
print("sample_submission path:", SAMPLE_SUB_PATH)
print("sample_submission shape:", sample_sub.shape)
print("columns[0:5]:", sample_sub.columns[:5].tolist())
sample_sub.head(3)


In [ ]:
submission = sample_sub.copy()
species_cols = [c for c in submission.columns if c != "row_id"]
if not species_cols:
    raise ValueError("sample_submission.csv does not contain prediction columns")

# Keep this as a true smoke test: valid file, minimal work, deterministic values.
submission[species_cols] = submission[species_cols].astype(np.float32)
submission[species_cols] = np.clip(submission[species_cols].fillna(0.0), 0.0, 1.0)

out_path = WORKDIR / "submission.csv"
submission.to_csv(out_path, index=False)

runtime_info = {
    "experiment_id": "exp_015g",
    "experiment_name": "smoke_submit",
    "competition_dir": str(COMPETITION_DIR),
    "sample_submission_path": str(SAMPLE_SUB_PATH),
    "rows": int(len(submission)),
    "n_species_cols": int(len(species_cols)),
    "wall_time_seconds": float(time.time() - _WALL_START),
    "kaggle_kernel_run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE"),
    "kaggle_url_base": os.environ.get("KAGGLE_URL_BASE"),
}
(WORKDIR / "smoke_submit_runtime.json").write_text(json.dumps(runtime_info, indent=2))

print("submission path:", out_path)
print("rows:", len(submission))
print("species cols:", len(species_cols))
print("wall time:", runtime_info["wall_time_seconds"])
print("Saved /kaggle/working/smoke_submit_runtime.json")
submission.head(3)


## Interpretation

- If this notebook times out in the same Kaggle settings, the problem is environmental or run-mode related.
- If this notebook succeeds, heavier notebooks should be debugged against the exact same settings and input attachments.
